# Enrich article metadata (IVDRS & SUDORS)

Collects richer metadata for the **39 journal articles** on the deposit list:

| Source | What it adds |
|---|---|
| **Scopus** (your API key + insttoken, on VPN) | publisher abstract, each author's own affiliation(s), ORCIDs, Scopus author IDs, author keywords, index terms, funding |
| **PubMed** | MeSH headings (with descriptor IDs and major-topic flags); fallback abstract/authors for articles not in Scopus |
| **Crossref / Unpaywall** | the publisher's license (e.g. CC BY 4.0); fallback funders and authors |
| **Prism staging** (optional, read-only) | how Prism's subject vocabulary stores each MeSH heading, so they can be added as proper MeSH subjects |

**Nothing is written to the deposit workbook.** Results go to `article_enrichment.xlsx`; upload that file to Claude,
who merges it into the workbook (keeping the child-community and theme keywords first, adding the rest after them).
Every raw API response is saved in `enrichment_raw/` so re-runs don't fetch anything twice.

Keep `enrich_articles.py` in the same folder as this notebook. Run on the VPN. Keys come from your `config.py`.

## Cell 1: Install (once)

In [3]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "requests", "openpyxl"], check=True)
print("Done.")

Done.


## Cell 2: Settings ✏️

Your keys are read from your `config.py` and never printed. It needs these lines (names can be changed below):

```python
SCOPUS_API_KEY = "..."
SCOPUS_INSTTOKEN = "..."
PRISM_STAGING_TOKEN = "..."    # only used for the optional Prism MeSH lookup
NCBI_API_KEY = "..."           # optional; makes PubMed faster
```
(A `.env` file also works: point `CONFIG_FILE` at it instead.)
`CONTACT_EMAIL` is sent to Crossref, Unpaywall and PubMed as their polite-use contact (Unpaywall requires it).

In [5]:
from pathlib import Path
import enrich_articles as ea

WORKBOOK  = Path("2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx")
CONFIG_FILE = Path("config.py")          # e.g. Path(r"C:\Users\keg827\Documents\Github_Repos\prism_project\config.py")
CONTACT_EMAIL = ""                       # ✏️ your Northwestern email

SCOPUS_KEY_VAR, INSTTOKEN_VAR, NCBI_KEY_VAR = "SCOPUS_API_KEY", "SCOPUS_INSTTOKEN", "NCBI_API_KEY"

OUT_XLSX  = Path("article_enrichment.xlsx")
RAW_DIR   = Path("enrichment_raw")
USE_CACHE = True                         # reuse saved responses; set False to fetch everything fresh

# Optional: look up how Prism stores each MeSH heading (read-only)
CHECK_PRISM_MESH = True
PRISM_URL        = "https://prism-staging.fsm.northwestern.edu"
PRISM_TOKEN_VAR  = "PRISM_STAGING_TOKEN"
VERIFY_SSL       = True                  # set False only if staging's certificate is expired again

env = ea.load_config(CONFIG_FILE)
print(f"{'found  ' if WORKBOOK.exists() else 'MISSING'}  {WORKBOOK.resolve()}")
print(f"{'found  ' if CONFIG_FILE.exists() else 'MISSING'}  {CONFIG_FILE.resolve()}")
for var in (SCOPUS_KEY_VAR, INSTTOKEN_VAR, NCBI_KEY_VAR, PRISM_TOKEN_VAR):
    print(f"  {var:<20} configured: {bool(env.get(var))}")
print(f"  CONTACT_EMAIL        set: {bool(CONTACT_EMAIL)}")

found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx
found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\config.py
  SCOPUS_API_KEY       configured: True
  SCOPUS_INSTTOKEN     configured: True
  NCBI_API_KEY         configured: True
  PRISM_STAGING_TOKEN  configured: True
  CONTACT_EMAIL        set: False


## Cell 3: Load the articles from the workbook

In [6]:
articles = ea.read_articles(WORKBOOK)
print(f"{len(articles)} journal articles on the Metadata for Deposit tab")
print(f"  with DOI: {sum(1 for a in articles if a['doi'])},  with PMID: {sum(1 for a in articles if a['pmid'])}")
client = ea.Client(env.get(SCOPUS_KEY_VAR, ""), env.get(INSTTOKEN_VAR, ""), CONTACT_EMAIL,
                   env.get(NCBI_KEY_VAR, ""), raw_dir=RAW_DIR)

39 journal articles on the Metadata for Deposit tab
  with DOI: 39,  with PMID: 37


## Cell 4: Test Scopus on one article

Checks your key, insttoken and VPN. **FULL** view is what gives per-author affiliations, ORCIDs and funding;
if it says META_ABS, you're not getting the full record (usually: not on VPN or insttoken missing).

In [7]:
test = next(a for a in articles if a["record_id"] == "IVSU-0069")      # Door-to-door overdose harm reduction
j, status = client.scopus(test["record_id"], test["doi"], use_cache=False)
print("Scopus:", status)
if j:
    sc = ea.parse_scopus(j)
    print("view:", sc["view"], "| title:", sc["title"])
    print("abstract:", (sc["abstract"][:200] + "...") if sc["abstract"] else "(none)")
    for a in sc["authors"][:5]:
        print(f"  {a['seq']}. {a['family']}, {a['given']}  ORCID:{a['orcid'] or '-'}  {' | '.join(a['affiliations'])[:110]}")
    print("author keywords:", sc["author_keywords"][:6])
    print("funding:", sc["funding"][:3])

Scopus: ok (FULL)
view: FULL | title: Door-to-door overdose harm reduction: an Illinois case study
abstract: Background: Harm reduction for people who use drugs (PWUD) is an established evidence-based practice that encompasses a wide variety of services, delivery formats, and settings and has been named a pr...
  1. Mason, Maryann  ORCID:-  Buehler Center for Health Policy and Economics, Institute for Public Health and Medicine, Northwestern Univers
  2. Johnson, Bruce  ORCID:-  NICASA Behavioral Health Services, 2031 Dugdale Road North Chicago, 60064, Waukegan, Illinois, IL, USA
  3. Schaffner, Christorpher  ORCID:-  JOLT Foundation, 2203 N. Sheridan Rd Peoria, 61604, Peoria, Illinois, IL, USA
  4. Johnston, Sean  ORCID:-  Buehler Center for Health Policy and Economics, Institute for Public Health and Medicine, Northwestern Univers
  5. Alexander, Ursula  ORCID:-  Buehler Center for Health Policy and Economics, Institute for Public Health and Medicine, Northwestern Univers
author keyw

## Cell 5: Collect everything for all articles

About 1–2 minutes. One line per article; `not in Scopus` is fine (PubMed/Crossref fill in).

In [8]:
results = ea.enrich(articles, client, use_cache=USE_CACHE)

[ 1/39] IVSU-0047  Scopus: ok (FULL)                    PubMed: ok         Crossref: ok     authors:6 MeSH:18 license:cc-by-4.0
[ 2/39] IVSU-0048  Scopus: ok (FULL)                    PubMed: ok         Crossref: ok     authors:5 MeSH:8 license:cc-by-nc-4.0
[ 3/39] IVSU-0049  Scopus: ok (FULL)                    PubMed: ok         Crossref: ok     authors:9 MeSH:6 license:cc-by-nc-4.0
[ 4/39] IVSU-0050  Scopus: ok (FULL)                    PubMed: ok         Crossref: ok     authors:8 MeSH:9 license:-
[ 5/39] IVSU-0051  Scopus: ok (FULL)                    PubMed: ok         Crossref: ok     authors:6 MeSH:10 license:-
[ 6/39] IVSU-0052  Scopus: ok (FULL)                    PubMed: ok         Crossref: ok     authors:6 MeSH:15 license:cc-by-nc-4.0
[ 7/39] IVSU-0053  Scopus: ok (FULL)                    PubMed: ok         Crossref: ok     authors:12 MeSH:17 license:-
[ 8/39] IVSU-0054  Scopus: ok (FULL)                    PubMed: ok         Crossref: ok     authors:11 MeSH:8 license:-
[

## Cell 6: Optional: how does Prism store these MeSH headings? (read-only)

In [9]:
if CHECK_PRISM_MESH:
    schemes = ea.prism_subject_lookup(results, PRISM_URL, env.get(PRISM_TOKEN_VAR, ""), verify=VERIFY_SSL)
    example = next((m for r in results for m in r["mesh"] if m.get("prism_id")), None)
    print("Example match:", example)
else:
    print("Skipped.")

   looked up 25/147 terms
   looked up 50/147 terms
   looked up 75/147 terms
   looked up 100/147 terms
   looked up 125/147 terms
Prism matched 137 of 147 distinct MeSH headings. Schemes seen: {'MeSH': 232, 'LCSH': 1263, 'LCNAF': 6}
Example match: {'descriptor': 'Humans', 'ui': 'D006801', 'major': False, 'qualifiers': [], 'prism_id': 'https://id.nlm.nih.gov/mesh/D006801', 'prism_scheme': 'MeSH'}


## Cell 7: Summary and things to look at

In [10]:
n = len(results)
print(f"Articles: {n}")
print(f"  in Scopus:            {sum(r['in_scopus'] for r in results)}  (FULL view: {sum(r['scopus_view'] == 'FULL' for r in results)})")
print(f"  with abstract:        {sum(bool(r['abstract']) for r in results)}")
print(f"  authors with affils:  {sum(1 for r in results for a in r['authors'] if a.get('affiliations'))} of {sum(len(r['authors']) for r in results)}")
print(f"  authors with ORCID:   {sum(1 for r in results for a in r['authors'] if a.get('orcid'))}")
print(f"  with MeSH:            {sum(bool(r['mesh']) for r in results)}   (headings: {sum(len(r['mesh']) for r in results)})")
print(f"  with license:         {sum(bool(r['license_id']) for r in results)}")
print(f"  with funding:         {sum(bool(r['funding']) for r in results)}")
print()
for r in results:
    issues = []
    if r["title_match"] is not None and r["title_match"] < 0.85:
        issues.append(f"title differs from source (similarity {r['title_match']}): {r['source_title'][:80]}")
    if not r["abstract"]:
        issues.append("no abstract found")
    if not r["license_id"]:
        issues.append(f"license not recognised: {r['license_url'] or 'none found'}")
    if not r["in_scopus"]:
        issues.append(f"not in Scopus (authors from {r['authors_source']})")
    if issues:
        print(r["record_id"], "\n    " + "\n    ".join(issues))

Articles: 39
  in Scopus:            37  (FULL view: 37)
  with abstract:        38
  authors with affils:  296 of 296
  authors with ORCID:   151
  with MeSH:            35   (headings: 360)
  with license:         15
  with funding:         26

IVSU-0050 
    license not recognised: none found
IVSU-0051 
    license not recognised: none found
IVSU-0053 
    license not recognised: none found
IVSU-0054 
    license not recognised: none found
IVSU-0068 
    license not recognised: none found
IVSU-0072 
    license not recognised: none found
IVSU-0073 
    license not recognised: none found
IVSU-0074 
    license not recognised: none found
IVSU-0075 
    license not recognised: none found
IVSU-0077 
    no abstract found
    license not recognised: none found
IVSU-0082 
    license not recognised: none found
IVSU-0083 
    license not recognised: none found
IVSU-0084 
    license not recognised: none found
IVSU-0085 
    license not recognised: none found
IVSU-0086 
    license not reco

## Cell 8: Save, then upload to Claude

Upload **`article_enrichment.xlsx`**. If cell 7 listed problems, or anything looks odd, also zip and upload the
`enrichment_raw` folder so Claude can read the original responses.

In [11]:
ea.save_excel(results, client.log, OUT_XLSX)
print("Saved", OUT_XLSX.resolve())

Saved C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\article_enrichment.xlsx
